In [1]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_csv(r'C:\Users\apaks\Desktop\Real Estate Project\artifacts\data\preprocessed-data\gurgaon_properties_post_feature_selection.csv')

In [15]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 83,1.18,3,3,2,relatively new,1612.0,0,0,unfurnished,budget,high-rise
1,flat,sector 62,4.00,3,3,3,new,2353.0,1,0,semifurnished,budget,high-rise
2,flat,sector 85,1.10,2,2,3+,relatively new,1459.0,0,1,unfurnished,luxury,low-rise
3,flat,sector 91,0.89,2,2,3+,relatively new,1297.0,0,0,semifurnished,budget,medium-rise
4,flat,sector 7,0.48,3,2,1,new,889.0,0,0,unfurnished,budget,low-rise


In [16]:
df.isnull().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
luxury_category    0
floor_category     0
dtype: int64

In [17]:
# input and output features

X = df.drop(columns = 'price')
y = df['price']


In [18]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 83,3,3,2,relatively new,1612.0,0,0,unfurnished,budget,high-rise
1,flat,sector 62,3,3,3,new,2353.0,1,0,semifurnished,budget,high-rise
2,flat,sector 85,2,2,3+,relatively new,1459.0,0,1,unfurnished,luxury,low-rise
3,flat,sector 91,2,2,3+,relatively new,1297.0,0,0,semifurnished,budget,medium-rise
4,flat,sector 7,3,2,1,new,889.0,0,0,unfurnished,budget,low-rise
...,...,...,...,...,...,...,...,...,...,...,...,...
3494,flat,sector 103,3,3,3+,relatively new,1627.0,0,0,semifurnished,luxury,low-rise
3495,house,sector 25,5,6,3+,relatively old,4518.0,1,1,unfurnished,luxury,low-rise
3496,flat,sector 108,2,2,2,under construction,1255.0,0,0,unfurnished,semi-luxury,low-rise
3497,flat,sector 104,3,4,3+,relatively new,2082.0,1,0,unfurnished,semi-luxury,high-rise


In [27]:
y_transformed = np.log1p(y)

In [28]:
y_transformed

0       0.779325
1       1.609438
2       0.741937
3       0.636577
4       0.392042
          ...   
3494    0.854415
3495    2.803360
3496    1.131402
3497    1.011601
3498    0.542324
Name: price, Length: 3499, dtype: float64

- Encoding used
    - categorical columns - ordinal encoding, onehot encoding, target encoding
    - numerical columns - StandardScaler

Ordinal Encoding

In [22]:
cat_columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']
num_columns_to_encode = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']

In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [35]:
# preprocessor for transformations

preprocessor = ColumnTransformer([
    ('categorical_columns', OrdinalEncoder(), cat_columns_to_encode),
    ('numerical_columns', StandardScaler(), num_columns_to_encode)
], remainder='passthrough')

In [36]:
# pipeline

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [37]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits = 10, shuffle= True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv = kfold, scoring = 'r2')

In [38]:
print(scores.mean(), scores.std())

0.7414147852957191 0.022792049372538363


In [34]:
# check mean absolute error

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

# train pipeline which includes transformation and model training
pipeline.fit(X_train, y_train)

# predictions
y_pred = pipeline.predict(X_test)

# mae
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
print(mae)

0.6607175935129945


In [45]:
# function to evaluate r2 score and mean absolute error

def scorer(model_name, model):

    output = []
    
    pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
    ])

    kfold = KFold(n_splits = 10, shuffle= True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv = kfold, scoring = 'r2')

    output.append(model_name)
    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
    
    output.append(mae)

    return output


In [49]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

In [55]:
model_dict = {
    'linear_regression' : LinearRegression(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'svr': SVR(),
    'decision_tree': DecisionTreeRegressor(),
    'extra_tree_regressor': ExtraTreeRegressor(),
    'random_forrest': RandomForestRegressor(),
    'gradient_boosting': GradientBoostingRegressor(),
    'ada_boost': AdaBoostRegressor(),
    'xgb_regressor': XGBRegressor(),
    'mlp': MLPRegressor()
}

In [56]:
model_outputs = []

for model_name, model in model_dict.items():
    model_outputs.append(scorer(model_name, model))
    print(model_name, 'completed')

linear_regression completed
ridge completed
lasso completed
svr completed
decision_tree completed
extra_tree_regressor completed
random_forrest completed
gradient_boosting completed
ada_boost completed
xgb_regressor completed
mlp completed


In [64]:
model_ordinal_encoding_results_df = pd.DataFrame(model_outputs, columns = ['model_name', 'r2_score', 'mae'])

In [72]:
model_ordinal_encoding_results_df.sort_values(by=['mae'])

,model_name,r2_score,mae
9,xgb_regressor,0.902986,0.519913
6,random_forrest,0.892107,0.531621
7,gradient_boosting,0.874655,0.599198
10,mlp,0.807683,0.711424
4,decision_tree,0.794122,0.732846
5,extra_tree_regressor,0.740246,0.807901
8,ada_boost,0.755462,0.811968
3,svr,0.765729,0.862447
0,linear_regression,0.741415,0.919793
1,ridge,0.741419,0.919884


- With ordinal encoding, tree based models are performing well. xgb regressor is performing the best followed by random forest and gradient boosting.
- linear models are doing the worst as expected in case of ordinal encoding

OneHot Encoding

In [77]:
cat_columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']       # cat columns to ordinal encode

cat_columns_to_ohe = ['sector', 'agePossession', 'furnishing_type']     # cat columns to ordinal encode

num_columns_to_encode = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']      # numerical columns to scale

In [ ]:
# preprocessor which includes onehot encoder
# will transfrom numerical columns first, then perform ordinal encoding in cat columns and then do ohe on some of them

preprocessor = ColumnTransformer([
    ('numerical_columns', StandardScaler(), num_columns_to_encode),
    ('cat_ord', OrdinalEncoder(), cat_columns_to_encode),
    ('cat_ohe', OneHotEncoder(drop='first'), cat_columns_to_ohe)
], remainder='passthrough')

In [79]:
model_outputs = []

for model_name, model in model_dict.items():
    model_outputs.append(scorer(model_name, model))
    print(model_name, 'completed')

linear_regression completed
ridge completed
lasso completed
svr completed
decision_tree completed
extra_tree_regressor completed
random_forrest completed
gradient_boosting completed
ada_boost completed
xgb_regressor completed


c:\Users\apaks\anaconda3\envs\realestatevenv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp completed


In [80]:
model_onehot_encoding_results_df = pd.DataFrame(model_outputs, columns = ['model_name', 'r2_score', 'mae'])

In [81]:
model_onehot_encoding_results_df.sort_values(by=['mae'])

,model_name,r2_score,mae
6,random_forrest,0.899514,0.503134
9,xgb_regressor,0.904118,0.532244
7,gradient_boosting,0.880754,0.578605
10,mlp,0.876212,0.586642
5,extra_tree_regressor,0.807104,0.647061
0,linear_regression,0.852726,0.666084
1,ridge,0.851754,0.667922
4,decision_tree,0.825580,0.672308
8,ada_boost,0.756852,0.837911
3,svr,0.775535,0.841204


- in case of xgb, the r2 score is slightly improved but the mae has gone up.
- In case of random forest, r2 score and mae are both improved
- linear models have performed better due to ohe as expected

Target Encoding

In [111]:
cat_columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']
num_columns_to_encode = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']

In [123]:
cat_columns_to_ohe = ['agePossession'] 
cat_column_to_target_encode = ['sector']

In [124]:
# preprocessor which also includes target encoding for sector column (high cardinality)

preprocessor = ColumnTransformer([
    ('numerical_columns', StandardScaler(), num_columns_to_encode),
    ('cat_ord', OrdinalEncoder(), cat_columns_to_encode),
    ('cat_ohe', OneHotEncoder(drop='first'), cat_columns_to_ohe),
    ('target_encode', TargetEncoder(), cat_column_to_target_encode)
], remainder='passthrough')

In [125]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [126]:
kfold = KFold(n_splits = 10, shuffle= True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv = kfold, scoring = 'r2')

In [127]:
scores.mean()

np.float64(0.8366449205675242)

In [128]:
scores.std()

np.float64(0.01726185592822723)

- Linear Regression shows improved results. Lets try with other algorithms as well

In [129]:
model_dict = {
    'linear_regression' : LinearRegression(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'svr': SVR(),
    'decision_tree': DecisionTreeRegressor(),
    'extra_tree_regressor': ExtraTreeRegressor(),
    'random_forrest': RandomForestRegressor(),
    'gradient_boosting': GradientBoostingRegressor(),
    'ada_boost': AdaBoostRegressor(),
    'xgb_regressor': XGBRegressor(),
    'mlp': MLPRegressor()
}

model_outputs = []

for model_name, model in model_dict.items():
    model_outputs.append(scorer(model_name, model))
    print(model_name, 'completed')

model_onehot_encoding_results_df = pd.DataFrame(model_outputs, columns = ['model_name', 'r2_score', 'mae'])

linear_regression completed
ridge completed
lasso completed
svr completed
decision_tree completed
extra_tree_regressor completed
random_forrest completed
gradient_boosting completed
ada_boost completed
xgb_regressor completed
mlp completed


In [130]:
model_onehot_encoding_results_df = pd.DataFrame(model_outputs, columns = ['model_name', 'r2_score', 'mae'])
model_onehot_encoding_results_df.sort_values(by= 'mae')

,model_name,r2_score,mae
6,random_forrest,0.901639,0.504929
9,xgb_regressor,0.900548,0.523032
7,gradient_boosting,0.887651,0.579765
10,mlp,0.854549,0.645426
4,decision_tree,0.827466,0.650771
5,extra_tree_regressor,0.790430,0.695640
1,ridge,0.836667,0.723311
0,linear_regression,0.836660,0.724499
8,ada_boost,0.823320,0.738867
3,svr,0.799155,0.796402


- Final conclusion
    - Tree based models, rf and xgboost gave best results
    - Target encoding (sector) gives better results